In [1]:
# Import the libraries
from floweaver import *
import pandas as pd
import re


In [2]:
%%html
<style>
.sankey{
    font-size: 24pt;
    font-family: sans-serif;

}
</style>

In [3]:
def break_string(x, words = 4):
        spaces = [i.start() for i in re.finditer(' ', x)]

        if len(spaces) >= words:
            return x[0].upper() + x[1:spaces[words - 1]] + '\n' + x[spaces[words - 1]+1:]
        else:
            return x[0].upper() + x[1:]

def get_Evalues_to_target(flows, process, m):
        if m == 'mass':
            value = round(sum(flows.loc[flows.target == process, 'value']) , 2)
            return r' (' + str(value) + ' Mt)'
        if m == 'energy':
            value = round(sum(flows.loc[flows.target == process, 'value']), 2)
            return ' (' + str(value) + ' PJ)'
        if m == '':
            value = round(sum(flows.loc[flows.target == process, 'value']), 2)
            return ' (' + str(value) + ')'

def get_Evalues_to_source(flows,process,m):
        if m == 'mass':
            value = round(sum(flows.loc[flows.source == process, 'value']) , 2)
            return ' (' + str(value) + ' Mt)'
        if m == 'energy':
            value = round(sum(flows.loc[(flows.source == process), 'value']), 2)
            # if len(flows.loc[(flows.source == process)&(flows.use==1)])>=1:
            #     return ' (' + str(value) + ' Mt)'
            # else:
            return ' (' + str(value) + ' PJ)'

        if m == '':
            value = round(sum(flows.loc[flows.source == process, 'value']), 2)
            return ' (' + str(value) + ')'

def get_Evalues_to_waypoint(flows,process,m,w):
    if m == 'mass':
        if w == 'device':
            value = round(sum(flows.loc[(flows.device == process), 'value']),2)
            return ' (' + str(value) + ' Mt)'
        if w == 'use':
            value = round(sum(flows.loc[(flows.use == process), 'value']),2)
            return ' (' + str(value) + ' Mt)'
        if w == 'system':
            value = round(sum(flows.loc[(flows.system == process), 'value']),2)
            return ' (' + str(value) + ' Mt)'
    if m == 'energy':
        if w == 'device':
            value = round(sum(flows.loc[(flows.device == process), 'value']),2)
            return ' (' + str(value) + ' PJ)'
        if w == 'use':
            value = round(sum(flows.loc[(flows.use == process), 'value']),2)
            return ' (' + str(value) + ' PJ)'
        if w == 'system':
            value = round(sum(flows.loc[(flows.system == process), 'value']),2)
            return ' (' + str(value) + ' PJ)'

### Here need to select the scenario to be visualized.
The data for all scenarios is in the same file, but in different sheets. The scenarios are: Reference, BAT, Electric, Hydrogen, Hybrid

In [4]:
# Upload the data
sheet = 'Hydrogen' # Scenarios: Reference, BAT, Electric, Hydrogen, Hybrid

flows = pd.read_excel(
    'data/SI-Data-Sankey.xlsx',
    sheet_name=sheet
)

flows['source']=flows['source'].replace('Natural Gas','Nat. Gas')
flows['system']=flows['system'].replace('Illuminated Space','Ill. Space')
# Replace the 2 by the unicode 2 as subscript
flows['device'] = [i.replace('H2', 'H₂') for i in flows['device']]
flows

,source,target,device,use,system,fuel,useful,lost,value,sector
0,Oil,Steel,Oil Burner,Heat,Furnace,oil,0.000000,0.000000,0.000000,steel
1,Oil,Steel,Oil Burner,Heat,Heated Space,oil,0.000000,0.000000,0.000000,steel
2,Oil,Steel,Oil Burner,Heat,Steam System,oil,0.000000,0.000000,0.000000,steel
3,Oil,Steel,Engine,Motion,Driven System,oil,0.000000,0.000000,0.000000,steel
4,Oil,Steel,Engine,Motion,Vehicle,oil,0.000000,0.000000,0.000000,steel
...,...,...,...,...,...,...,...,...,...,...
445,Hydrogen,Other,H₂ Burner,Heat,Steam System,hydrogen,42.486297,4.720700,47.206996,other
446,Hydrogen,Other,Engine,Motion,Driven System,hydrogen,3.863111,5.120869,8.983980,other
447,Hydrogen,Other,Engine,Motion,Vehicle,hydrogen,0.000000,0.000000,0.000000,other
448,Hydrogen,Other,Lighting Device,Other,Ill. Space,hydrogen,0.000000,0.000000,0.000000,other


In [5]:
nodes={
    # Fuels
    'Fuels':ProcessGroup(['Hydrogen','Nat. Gas', 'Electricity','Oil','Coal','Biomass','Heat'],
                                partition = Partition(tuple([
                                                            Group(break_string(i, words=3) + get_Evalues_to_source(flows, i,'energy'),
                                                                  (('source', (i,)),))
                                                            for i in ['Hydrogen','Nat. Gas','Oil','Coal', 'Biomass','Heat','Electricity']
                                                                ])),
                         title='Fuels (' + str(round(sum(flows.value) , 1)) + ' PJ)'),


    'Devices':Waypoint(partition = Partition(tuple([
                                                        Group(break_string(i, words=3) + get_Evalues_to_waypoint(flows, i,'energy','device'),
                                                                  (('device', (i,)),))
                                                            for i in ['H\u2082 Burner','Gas Burner','Oil Burner','Coal Burner','Biomass Burner', 'Engine', 'Biomass Engine', 'Gas Engine', 'Coal Engine', 'Heat Exchanger', 'Electric Heater', 'Electric Motor', 'Electronic', 'Lighting Device']
                                                                ])),
                           title='Devices'),

    'Use':Waypoint(partition = Partition(tuple([
                                                        Group(break_string(i, words=3) + get_Evalues_to_waypoint(flows, i,'energy','use'),
                                                                  (('use', (i,)),))
                                                            for i in list(flows['use'].unique())
                                                                ])),
                           title='Use'),

     'Systems':Waypoint(partition = Partition(tuple([
                                                        Group(break_string(i, words=3) + get_Evalues_to_waypoint(flows, i,'energy','system'),
                                                                  (('system', (i,)),))
                                                            for i in list(flows['system'].unique())
                                                                ])),
                           title='Passive Systems'),


    'Sector':ProcessGroup(list(flows['target'].unique()),
                                partition = Partition(tuple([
                                                            Group(break_string(i, words=3) + get_Evalues_to_target(flows, i,'energy'),
                                                                  (('target', (i,)),))
                                                            for i in list(flows['target'].unique())
                                                                ])), title='Sector'),

}



ordering=[
        [['Fuels']],
        [['Devices']],
        [['Use']],
        [['Systems']],
        [['Sector']],
          # [['Out_M'],['Out_C'],['Out_E']],
          ]

bundles=[
        Bundle('Fuels','Sector',waypoints=['Devices','Use','Systems']),
    ]

# Color palette
palette={
         'biomass':'#1b7837',
         'oil':'#49006a',
         'coal':'#1a1a1a',
         'electricity':'#fed976',
         'heat':'tomato',
         'gas':'#bababa',
         'hydrogen':'royalblue',
         'Heat H':'darkred',
         'Heat R':'deeppink',
         'Reaction':'orange',
         'Oxygen':'deepskyblue',
         'Water':'dodgerblue',
         'Process':'slategray', # Color for CO2 Chemical reactions
         'Energy':'darkslategray', # Color CO2 Fuels
         'Energy O':'teal', # Color CO2 Fuels Organic
         'W':'#FF000000'}

# Partitions
by_type = Partition.Simple('fuel',flows['fuel'].unique())


# Update the SDD with the new nodes, ordering & bundles.
sdd = SankeyDefinition(nodes, bundles, ordering,flow_partition=by_type)
H=1050
w1=weave(sdd,flows,palette=palette).to_widget( width=H*2, height=H,margins=dict(left=350, right=350,top=0,bottom=0))
w1.scale=0.6
# w1.auto_save_png(f'Sankey_{sheet}.png')
w1


SankeyWidget(groups=[{'id': 'Fuels', 'type': 'process', 'title': 'Fuels (829.7 PJ)', 'nodes': ['Fuels^Hydrogen…

### Here we can select the sector to be visualized.
The sectors are: steel, aluminium, machinery, cement, ceramics, glass, chemicals, paper, food, other

In [6]:
# Sectors: 'steel', 'aluminium', 'machinery', 'cement', 'ceramics', 'glass',
# 'chemicals', 'paper', 'food', 'other'
sector = 'steel'
sector_flows = flows[flows['sector'] == sector]

In [7]:
nodes = {
    # Fuels
    'Fuels': ProcessGroup(['Hydrogen','Nat. Gas', 'Electricity', 'Oil', 'Coal', 'Biomass', 'Heat'],
                          partition=Partition(tuple([
                              Group(break_string(i, words=3) + get_Evalues_to_source(sector_flows, i, 'energy'),
                                    (('source', (i,)),))
                              for i in ['Hydrogen','Nat. Gas', 'Oil', 'Coal', 'Biomass', 'Heat', 'Electricity']
                          ])),
                          title='Fuels (' + str(round(sum(sector_flows.value), 1)) + ' PJ)'),

    'Devices': Waypoint(partition=Partition(tuple([
        Group(break_string(i, words=3) + get_Evalues_to_waypoint(sector_flows, i, 'energy', 'device'),
              (('device', (i,)),))
        for i in ['H\u2082 Burner','Gas Burner', 'Oil Burner', 'Coal Burner', 'Biomass Burner', 'Engine', 'Biomass Engine',
                  'Gas Engine', 'Coal Engine', 'Heat Exchanger', 'Electric Heater', 'Electric Motor',
                  'Electronic', 'Lighting Device']
    ])),
        title='Devices'),

    'Use': Waypoint(partition=Partition(tuple([
        Group(break_string(i, words=3) + get_Evalues_to_waypoint(sector_flows, i, 'energy', 'use'),
              (('use', (i,)),))
        for i in list(sector_flows['use'].unique())
    ])),
        title='Use'),

    'Systems': Waypoint(partition=Partition(tuple([
        Group(break_string(i, words=3) + get_Evalues_to_waypoint(sector_flows, i, 'energy', 'system'),
              (('system', (i,)),))
        for i in list(sector_flows['system'].unique())
    ])),
        title='Passive Systems'),

    'Sector': ProcessGroup(list(sector_flows['target'].unique()),
                           partition=Partition(tuple([
                               Group(break_string(i, words=3) + get_Evalues_to_target(sector_flows, i, 'energy'),
                                     (('target', (i,)),))
                               for i in list(sector_flows['target'].unique())
                           ])), title='Sector'),

}

ordering = [
    [['Fuels']],
    [['Devices']],
    [['Use']],
    [['Systems']],
    [['Sector']],
]

bundles = [
    Bundle('Fuels', 'Sector', waypoints=['Devices', 'Use', 'Systems']),
]

# Color palette
palette = {
    'biomass': '#1b7837',
    'oil': '#49006a',
    'coal': '#1a1a1a',
    'electricity': '#fed976',
    'heat': 'tomato',
    'gas': '#bababa',
    'hydrogen':'royalblue',
    'Heat H': 'darkred',
    'Heat R': 'deeppink',
    'Reaction': 'orange',
    'Oxygen': 'deepskyblue',
    'Water': 'dodgerblue',
    'Process': 'slategray',  # Color for CO2 Chemical reactions
    'Energy': 'darkslategray',  # Color CO2 Fuels
    'Energy O': 'teal',  # Color CO2 Fuels Organic
    'W': '#FF000000'
}
by_type = Partition.Simple('fuel', sector_flows['fuel'].unique())

case = 'Hybrid' #'Actual' 'BAT' 'Electric' 'Hydrogen' 'Hybrid'
print(sector)
# Update the SDD with the new nodes, ordering & bundles.
H = 1050
sdd = SankeyDefinition(nodes, bundles, ordering, flow_partition=by_type)
w2=weave(sdd, sector_flows, palette=palette).to_widget(width=H*2, height=H, margins=dict(left=350, right=350, top=0, bottom=0))
w2.scale=2
# w2.auto_save_png(f'Sankey_{sheet}_{sector}.png')
w2


steel


SankeyWidget(groups=[{'id': 'Fuels', 'type': 'process', 'title': 'Fuels (41.9 PJ)', 'nodes': ['Fuels^Hydrogen …